# 02 — Primary direct-biomarker network dataset

Builds the analytical dataset for the 29 primary biomarkers: winsorization (1st/99th percentile), log1p for right-skewed non-negative variables, median imputation, and z-score standardization (Methods 2.5, 2.6).

## Environment setup

In [ ]:
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/<your-org>/<your-repo>.git"  # TODO: set to the published GitHub repo URL
REPO_DIR_NAME = "pcos_network_architecture_jcem_v3_jcem_10of10"


def _in_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False


if _in_colab():
    PROJECT_ROOT = Path("/content") / REPO_DIR_NAME
    if not PROJECT_ROOT.exists():
        subprocess.run(["git", "clone", REPO_URL, str(PROJECT_ROOT)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                     "scikit-learn", "networkx", "statsmodels", "openpyxl"], check=True)
else:
    PROJECT_ROOT = Path.cwd().resolve()
    while not ((PROJECT_ROOT / "data_raw").exists() and (PROJECT_ROOT / "outputs").exists()):
        if PROJECT_ROOT.parent == PROJECT_ROOT:
            break
        PROJECT_ROOT = PROJECT_ROOT.parent

os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))
from scripts._pipeline_common import *  # noqa: F401,F403

print("Project root:", PROJECT_ROOT)


In [ ]:
import json
import pandas as pd

in_dir = PROJECT_ROOT / 'outputs' / '01_ingest_harmonize'
H = pd.read_csv(in_dir / 'harmonized_features.csv')
fd = pd.read_csv(in_dir / 'feature_dictionary_v2.csv')
primary = fd.loc[fd.included_primary, 'feature'].tolist()

X = H[primary]
Y, Yi, Z, T = preprocess(X)

out_dir = PROJECT_ROOT / 'outputs' / '02_primary_network_dataset'
out_dir.mkdir(parents=True, exist_ok=True)
Y.to_csv(out_dir / 'primary_direct_features_unimputed.csv', index=False)
Yi.to_csv(out_dir / 'primary_direct_features_imputed.csv', index=False)
Z.to_csv(out_dir / 'primary_direct_features_z.csv', index=False)
T.assign(domain=lambda d: d.feature.map(DOM)).to_csv(out_dir / 'primary_preprocessing_transformations.csv', index=False)
pd.DataFrame([{'feature': f, 'domain': DOM[f]} for f in primary]).to_csv(out_dir / 'primary_feature_set.csv', index=False)
(out_dir / 'primary_space_definitions.json').write_text(
    json.dumps({d: [f for f in primary if DOM[f] == d] for d in sorted(set(DOM[f] for f in primary))}, indent=2),
    encoding='utf-8',
)

print(f'Participants: {len(Z)}; primary biomarkers: {len(primary)}')
Z.describe().T.head()
